# AutoRA Workflow

Generated by AutoRA Workflow Editor on 2026-08-10T05:13:22.860Z

## 1. Install dependencies

In [ ]:
%pip install autora-synthetic==2.2.0 autora-core==5.0.3 autora-theorist-bsr==1.0.0

## 2. Imports

In [ ]:
from autora.state import on_state, Delta, estimator_on_state, StandardState
from autora.variable import VariableCollection
from autora.experimentalist.random import pool as random_pooler, sample as random_sampler
from autora.experiment_runner.synthetic.economics.expected_value_theory import expected_value_theory
from autora.theorist.bsr.regressor import BSRRegressor

import pandas as pd

## 3. Component definitions

In [ ]:
# Random Pooler
@on_state()
def random_pooler_on_state(variables: VariableCollection) -> Delta:
    return Delta(conditions=random_pooler(variables, num_samples=5, replace=True))

In [ ]:
# Random Sampler
@on_state()
def random_sampler_on_state(conditions: pd.DataFrame, num_samples: int = 1) -> Delta:
    return Delta(conditions=random_sampler(conditions=conditions, num_samples=num_samples, replace=False))

In [ ]:
# Expected Value Theory (Synthetic, Economics)
runner = expected_value_theory(choice_temperature=0.1, value_lambda=0.5, resolution=10, minimum_value=-1, maximum_value=1)

@on_state()
def expected_value_theory_on_state(conditions: pd.DataFrame) -> Delta:
    return Delta(experiment_data=runner.run(conditions=conditions, added_noise=0.01))

In [ ]:
# BSR Regressor
bsr_regressor_on_state = estimator_on_state(BSRRegressor(tree_num=3, itr_num=5000, alpha1=0.4, alpha2=0.4, beta=-1, show_log=False, val=100, last_idx=-1, prior_name="Uniform"))

## 4. Run the workflow

In [ ]:
# Variables are governed by the experiment runner defined above
assert runner.variables is not None
variables = runner.variables

# Initialize state
state = StandardState(variables=variables)

# Main experiment loop (1 cycles)
for i in range(1):
    print(f'Cycle {i}')

    # Random Pooler
    state = random_pooler_on_state(state)

    # Random Sampler
    state = random_sampler_on_state(state, num_samples=1)

    # Expected Value Theory (Synthetic, Economics)
    state = expected_value_theory_on_state(state)

    # BSR Regressor
    state = bsr_regressor_on_state(state)


print("Workflow completed!")
state